In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Conv1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# =========================
# Configuration and constants
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100          # time-normalized points per cycle (101 for exact 0-100%)
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']
FEATURE_COLUMNS = [
    'IMU101_v0', 'IMU101_v1',
    'IMU103_v0', 'IMU103_v1',
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1',
    #'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6',
    #'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]

# Model hyperparameters
UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32

# Checkpoint naming (kept distinct from the old padded-input runs)
WEIGHTS_PREFIX = 'best_timenorm'


def resample_cycle(features, target_len=TARGET_LEN):
    """Time-normalize one cycle onto a fixed number of points (0-100% of cycle)."""
    n = features.shape[0]
    kind = 'cubic' if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_and_split_data():
    """Load cycles, select features, time-normalize, encode labels, and split."""
    raw = []

    if not os.path.exists(ROOT_DIR):
        return None, None, None, None, 0, 0, None

    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                file_path = os.path.join(root, filename)

                # Parse label and cycle id from filename
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cycle_id_str = raw_id_part.split('_')[-1]

                if label not in VALID_LABELS or not cycle_id_str.isdigit():
                    continue

                try:
                    df = pd.read_csv(file_path)
                except Exception:
                    continue

                # Feature selection
                if any(col not in df.columns for col in FEATURE_COLUMNS):
                    continue
                df = df[FEATURE_COLUMNS]
                cycle_features = df.values
                if cycle_features.shape[0] < 2 or cycle_features.shape[1] == 0:
                    continue

                raw.append((cycle_features, label, file_path))

    if not raw:
        return None, None, None, None, 0, 0, None

    # Length distribution + outlier guard (drop mis-segmented cycles)
    lengths = np.array([r[0].shape[0] for r in raw])
    med = np.median(lengths)
    print(f"lengths: min={lengths.min()} med={med:.0f} max={lengths.max()} n={len(raw)}")

    keep = [r for r in raw if 0.5 * med <= r[0].shape[0] <= 2.0 * med]
    print(f"dropped {len(raw) - len(keep)} outlier cycles")

    # Time-normalize every cycle to TARGET_LEN points
    X_full = np.array([resample_cycle(r[0]) for r in keep])
    y_full = np.array([r[1] for r in keep])
    ids_full = np.array([r[2] for r in keep])

    # Encode labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_full)
    y_encoded = to_categorical(y_int)
    class_names = le.classes_

    num_features = X_full.shape[2]
    num_classes = y_encoded.shape[1]

    # Train/test split by unique file paths (cycle-level split)
    unique_ids = np.unique(ids_full)
    train_ids, test_ids = train_test_split(unique_ids, train_size=TRAIN_RATIO, random_state=42, shuffle=True)
    train_idx = np.where(np.isin(ids_full, train_ids))[0]
    test_idx = np.where(np.isin(ids_full, test_ids))[0]

    X_train, X_test = X_full[train_idx], X_full[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    return X_train, X_test, y_train, y_test, num_features, num_classes, class_names


def build_model(model_type, sequence_length, num_features, num_classes):
    """Build and compile a model of the requested type: 'lstm', 'gru', or 'cnn'."""
    if model_type == 'lstm':
        model = Sequential([
            LSTM(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='LSTM'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'gru':
        model = Sequential([
            GRU(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='GRU'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'cnn':
        model = Sequential([
            # Convolution over time steps
            Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(sequence_length, num_features)),
            Dropout(DROPOUT_RATE),
            Conv1D(filters=64, kernel_size=5, activation='relu'),
            GlobalAveragePooling1D(),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def train_and_compare():
    """Train and evaluate multiple architectures on the same split."""
    X_train, X_test, y_train, y_test, num_features, num_classes, class_names = load_and_split_data()
    if X_train is None or X_train.shape[0] < 1:
        print("Insufficient data; training aborted.")
        return

    sequence_length = X_train.shape[1]
    model_types = ['lstm', 'gru', 'cnn']

    for mtype in model_types:
        print(f"\n=== Training {mtype.upper()} model ===")
        model = build_model(mtype, sequence_length, num_features, num_classes)
        weights_path = f'{WEIGHTS_PREFIX}_{mtype}_IMU_{sequence_length}.weights.h5'

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
            ModelCheckpoint(weights_path, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=0)
        ]

        history = model.fit(
            X_train, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_test, y_test),
            callbacks=callbacks,
            verbose=1
        )

        # Load best weights and evaluate
        try:
            model.load_weights(weights_path)
        except Exception as e:
            print(f"Warning: Could not load best weights for {mtype}: {e}")

        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"{mtype.upper()} Test Loss: {loss:.4f} | Test Accuracy: {accuracy:.4f}")

        # Classification report
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        y_true = np.argmax(y_test, axis=1)
        print(f"\n{mtype.upper()} Classification Report:")
        print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


if __name__ == '__main__':
    tf.get_logger().setLevel('INFO')
    train_and_compare()

2026-07-23 16:48:14.877140: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784839694.892063 2471052 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784839694.896733 2471052 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784839694.908361 2471052 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784839694.908372 2471052 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784839694.908374 2471052 computation_placer.cc:177] computation placer alr

lengths: min=81 med=138 max=250 n=5166
dropped 0 outlier cycles

=== Training LSTM model ===


I0000 00:00:1784839709.517044 2471052 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 45848 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1784839709.522551 2471052 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 46677 MB memory:  -> device: 1, name: NVIDIA RTX A6000, pci bus id: 0000:25:00.0, compute capability: 8.6
I0000 00:00:1784839709.524049 2471052 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 46677 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1784839709.525447 2471052 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 46677 MB memory:  -> device: 3, name: NVIDIA RTX A6000, pci bus id: 0000:c1:00.0, compute capability: 8.6
/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWar

Epoch 1/25


I0000 00:00:1784839711.659952 2471880 cuda_dnn.cc:529] Loaded cuDNN version 90701


...

130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3975 - loss: 1.3435 - val_accuracy: 0.6586 - val_loss: 0.9127
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6033 - loss: 0.9594 - val_accuracy: 0.7544 - val_loss: 0.7197
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6992 - loss: 0.7765 - val_accuracy: 0.8317 - val_loss: 0.5602
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7685 - loss: 0.6270 - val_accuracy: 0.8453 - val_loss: 0.4864
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8141 - loss: 0.5276 - val_accuracy: 0.8482 - val_loss: 0.4421
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8373 - loss: 0.4490 - val_accuracy: 0.8578 - val_loss: 0.3991
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8405 - loss: 0.4273 - val_accuracy: 0.8936 - val_loss: 0.3299
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8688 - loss: 0.3532 - val_accuracy: 0.9197 - val

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1784839773.048022 2471876 service.cc:152] XLA service 0x15331d70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784839773.048090 2471876 service.cc:160]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1784839773.048096 2471876 service.cc:160]   StreamExecutor device (1): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1784839773.048098 2471876 service.cc:160]   StreamExecutor device (2): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1784839773.048100 2471876 service.cc:160]   StreamExecutor device (3): NVIDIA RTX A6000, Compute Capab

115/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3705 - loss: 6.1570  

I0000 00:00:1784839774.731939 2471876 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


130/130 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.3852 - loss: 5.6966 - val_accuracy: 0.7070 - val_loss: 0.9460
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7130 - loss: 0.6941 - val_accuracy: 0.8472 - val_loss: 0.5831
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8001 - loss: 0.4946 - val_accuracy: 0.8559 - val_loss: 0.4478
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8525 - loss: 0.4003 - val_accuracy: 0.9226 - val_loss: 0.2871
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8658 - loss: 0.3565 - val_accuracy: 0.8946 - val_loss: 0.3026
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8693 - loss: 0.3565 - val_accuracy: 0.9178 - val_loss: 0.2340
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8996 - loss: 0.2696 - val_accuracy: 0.9458 - val_loss: 0.1753
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9163 - loss: 0.2425 - val_accuracy: 0.9439 - val

In [2]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Conv1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# =========================
# Configuration and constants
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100          # time-normalized points per cycle (101 for exact 0-100%)
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']
FEATURE_COLUMNS = [
    'IMU101_v0', 'IMU101_v1',
    'IMU103_v0', 'IMU103_v1',
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1',
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6',
    'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]

# Model hyperparameters
UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32

# Checkpoint naming (kept distinct from the old padded-input runs)
WEIGHTS_PREFIX = 'best_timenorm'


def resample_cycle(features, target_len=TARGET_LEN):
    """Time-normalize one cycle onto a fixed number of points (0-100% of cycle)."""
    n = features.shape[0]
    kind = 'cubic' if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_and_split_data():
    """Load cycles, select features, time-normalize, encode labels, and split."""
    raw = []

    if not os.path.exists(ROOT_DIR):
        return None, None, None, None, 0, 0, None

    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                file_path = os.path.join(root, filename)

                # Parse label and cycle id from filename
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cycle_id_str = raw_id_part.split('_')[-1]

                if label not in VALID_LABELS or not cycle_id_str.isdigit():
                    continue

                try:
                    df = pd.read_csv(file_path)
                except Exception:
                    continue

                # Feature selection
                if any(col not in df.columns for col in FEATURE_COLUMNS):
                    continue
                df = df[FEATURE_COLUMNS]
                cycle_features = df.values
                if cycle_features.shape[0] < 2 or cycle_features.shape[1] == 0:
                    continue

                raw.append((cycle_features, label, file_path))

    if not raw:
        return None, None, None, None, 0, 0, None

    # Length distribution + outlier guard (drop mis-segmented cycles)
    lengths = np.array([r[0].shape[0] for r in raw])
    med = np.median(lengths)
    print(f"lengths: min={lengths.min()} med={med:.0f} max={lengths.max()} n={len(raw)}")

    keep = [r for r in raw if 0.5 * med <= r[0].shape[0] <= 2.0 * med]
    print(f"dropped {len(raw) - len(keep)} outlier cycles")

    # Time-normalize every cycle to TARGET_LEN points
    X_full = np.array([resample_cycle(r[0]) for r in keep])
    y_full = np.array([r[1] for r in keep])
    ids_full = np.array([r[2] for r in keep])

    # Encode labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_full)
    y_encoded = to_categorical(y_int)
    class_names = le.classes_

    num_features = X_full.shape[2]
    num_classes = y_encoded.shape[1]

    # Train/test split by unique file paths (cycle-level split)
    unique_ids = np.unique(ids_full)
    train_ids, test_ids = train_test_split(unique_ids, train_size=TRAIN_RATIO, random_state=42, shuffle=True)
    train_idx = np.where(np.isin(ids_full, train_ids))[0]
    test_idx = np.where(np.isin(ids_full, test_ids))[0]

    X_train, X_test = X_full[train_idx], X_full[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    return X_train, X_test, y_train, y_test, num_features, num_classes, class_names


def build_model(model_type, sequence_length, num_features, num_classes):
    """Build and compile a model of the requested type: 'lstm', 'gru', or 'cnn'."""
    if model_type == 'lstm':
        model = Sequential([
            LSTM(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='LSTM'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'gru':
        model = Sequential([
            GRU(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='GRU'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'cnn':
        model = Sequential([
            # Convolution over time steps
            Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(sequence_length, num_features)),
            Dropout(DROPOUT_RATE),
            Conv1D(filters=64, kernel_size=5, activation='relu'),
            GlobalAveragePooling1D(),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def train_and_compare():
    """Train and evaluate multiple architectures on the same split."""
    X_train, X_test, y_train, y_test, num_features, num_classes, class_names = load_and_split_data()
    if X_train is None or X_train.shape[0] < 1:
        print("Insufficient data; training aborted.")
        return

    sequence_length = X_train.shape[1]
    model_types = ['lstm', 'gru', 'cnn']

    for mtype in model_types:
        print(f"\n=== Training {mtype.upper()} model ===")
        model = build_model(mtype, sequence_length, num_features, num_classes)
        weights_path = f'{WEIGHTS_PREFIX}_{mtype}_IMU+Pressure_{sequence_length}.weights.h5'

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
            ModelCheckpoint(weights_path, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=0)
        ]

        history = model.fit(
            X_train, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_test, y_test),
            callbacks=callbacks,
            verbose=1
        )

        # Load best weights and evaluate
        try:
            model.load_weights(weights_path)
        except Exception as e:
            print(f"Warning: Could not load best weights for {mtype}: {e}")

        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"{mtype.upper()} Test Loss: {loss:.4f} | Test Accuracy: {accuracy:.4f}")

        # Classification report
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        y_true = np.argmax(y_test, axis=1)
        print(f"\n{mtype.upper()} Classification Report:")
        print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


if __name__ == '__main__':
    tf.get_logger().setLevel('INFO')
    train_and_compare()

lengths: min=81 med=138 max=250 n=5166
dropped 0 outlier cycles

=== Training LSTM model ===
Epoch 1/25


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - accuracy: 0.4241 - loss: 1.2711 - val_accuracy: 0.7872 - val_loss: 0.7414
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7114 - loss: 0.7677 - val_accuracy: 0.8830 - val_loss: 0.4404
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8499 - loss: 0.4489 - val_accuracy: 0.9110 - val_loss: 0.3246
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8871 - loss: 0.3612 - val_accuracy: 0.9449 - val_loss: 0.2355
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9125 - loss: 0.2920 - val_accuracy: 0.9574 - val_loss: 0.1760
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9287 - loss: 0.2371 - val_accuracy: 0.9603 - val_loss: 0.1545
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9374 - loss: 0.2063 - val_accuracy: 0.9720 - val_loss: 0.1208
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9470 - loss: 0.1659 - val_accuracy: 0.9681 - val

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3413 - loss: 1.6684 - val_accuracy: 0.8182 - val_loss: 0.6968
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7110 - loss: 0.7403 - val_accuracy: 0.8965 - val_loss: 0.4391
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8248 - loss: 0.4862 - val_accuracy: 0.9236 - val_loss: 0.3219
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8876 - loss: 0.3510 - val_accuracy: 0.9429 - val_loss: 0.2405
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9191 - loss: 0.2594 - val_accuracy: 0.9574 - val_loss: 0.1880
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9392 - loss: 0.2184 - val_accuracy: 0.9594 - val_loss: 0.1588
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.9381 - loss: 0.2005 - val_accuracy: 0.9632 - val_loss: 0.1403
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9537 - loss: 0.1541 - val_accuracy: 0.9642 - val

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.6213 - loss: 3.3730 - val_accuracy: 0.9178 - val_loss: 0.2738
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8882 - loss: 0.2904 - val_accuracy: 0.9565 - val_loss: 0.1856
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9336 - loss: 0.1855 - val_accuracy: 0.9623 - val_loss: 0.1372
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9505 - loss: 0.1487 - val_accuracy: 0.9739 - val_loss: 0.0955
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9493 - loss: 0.1341 - val_accuracy: 0.9778 - val_loss: 0.1076
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9610 - loss: 0.1094 - val_accuracy: 0.9739 - val_loss: 0.0940
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9670 - loss: 0.0964 - val_accuracy: 0.9845 - val_loss: 0.0820
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9668 - loss: 0.0944 - val_accuracy: 0.9826 - val

In [3]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Conv1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# =========================
# Configuration and constants
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100          # time-normalized points per cycle (101 for exact 0-100%)
INTERP_KIND = 'cubic'     # 'linear' avoids overshoot on sharp pressure transitions
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']
FEATURE_COLUMNS = [
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6',
    'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]

# Model hyperparameters
UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32

# Checkpoint naming (kept distinct from the old padded-input runs)
WEIGHTS_PREFIX = 'best_timenorm'


def resample_cycle(features, target_len=TARGET_LEN):
    """Time-normalize one cycle onto a fixed number of points (0-100% of cycle)."""
    n = features.shape[0]
    kind = INTERP_KIND if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_and_split_data():
    """Load cycles, select features, time-normalize, encode labels, and split."""
    raw = []

    if not os.path.exists(ROOT_DIR):
        return None, None, None, None, 0, 0, None

    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                file_path = os.path.join(root, filename)

                # Parse label and cycle id from filename
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cycle_id_str = raw_id_part.split('_')[-1]

                if label not in VALID_LABELS or not cycle_id_str.isdigit():
                    continue

                try:
                    df = pd.read_csv(file_path)
                except Exception:
                    continue

                # Feature selection
                if any(col not in df.columns for col in FEATURE_COLUMNS):
                    continue
                df = df[FEATURE_COLUMNS]
                cycle_features = df.values
                if cycle_features.shape[0] < 2 or cycle_features.shape[1] == 0:
                    continue

                raw.append((cycle_features, label, file_path))

    if not raw:
        return None, None, None, None, 0, 0, None

    # Length distribution + outlier guard (drop mis-segmented cycles)
    lengths = np.array([r[0].shape[0] for r in raw])
    med = np.median(lengths)
    print(f"lengths: min={lengths.min()} med={med:.0f} max={lengths.max()} n={len(raw)}")

    keep = [r for r in raw if 0.5 * med <= r[0].shape[0] <= 2.0 * med]
    print(f"dropped {len(raw) - len(keep)} outlier cycles")

    # Time-normalize every cycle to TARGET_LEN points
    X_full = np.array([resample_cycle(r[0]) for r in keep])
    y_full = np.array([r[1] for r in keep])
    ids_full = np.array([r[2] for r in keep])

    # Encode labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_full)
    y_encoded = to_categorical(y_int)
    class_names = le.classes_

    num_features = X_full.shape[2]
    num_classes = y_encoded.shape[1]

    # Train/test split by unique file paths (cycle-level split)
    unique_ids = np.unique(ids_full)
    train_ids, test_ids = train_test_split(unique_ids, train_size=TRAIN_RATIO, random_state=42, shuffle=True)
    train_idx = np.where(np.isin(ids_full, train_ids))[0]
    test_idx = np.where(np.isin(ids_full, test_ids))[0]

    X_train, X_test = X_full[train_idx], X_full[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    return X_train, X_test, y_train, y_test, num_features, num_classes, class_names


def build_model(model_type, sequence_length, num_features, num_classes):
    """Build and compile a model of the requested type: 'lstm', 'gru', or 'cnn'."""
    if model_type == 'lstm':
        model = Sequential([
            LSTM(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='LSTM'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'gru':
        model = Sequential([
            GRU(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='GRU'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'cnn':
        model = Sequential([
            # Convolution over time steps
            Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(sequence_length, num_features)),
            Dropout(DROPOUT_RATE),
            Conv1D(filters=64, kernel_size=5, activation='relu'),
            GlobalAveragePooling1D(),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def train_and_compare():
    """Train and evaluate multiple architectures on the same split."""
    X_train, X_test, y_train, y_test, num_features, num_classes, class_names = load_and_split_data()
    if X_train is None or X_train.shape[0] < 1:
        print("Insufficient data; training aborted.")
        return

    sequence_length = X_train.shape[1]
    model_types = ['lstm', 'gru', 'cnn']

    for mtype in model_types:
        print(f"\n=== Training {mtype.upper()} model ===")
        model = build_model(mtype, sequence_length, num_features, num_classes)
        weights_path = f'{WEIGHTS_PREFIX}_{mtype}_pressure_{sequence_length}.weights.h5'

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
            ModelCheckpoint(weights_path, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=0)
        ]

        history = model.fit(
            X_train, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_test, y_test),
            callbacks=callbacks,
            verbose=1
        )

        # Load best weights and evaluate
        try:
            model.load_weights(weights_path)
        except Exception as e:
            print(f"Warning: Could not load best weights for {mtype}: {e}")

        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"{mtype.upper()} Test Loss: {loss:.4f} | Test Accuracy: {accuracy:.4f}")

        # Classification report
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        y_true = np.argmax(y_test, axis=1)
        print(f"\n{mtype.upper()} Classification Report:")
        print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


if __name__ == '__main__':
    tf.get_logger().setLevel('INFO')
    train_and_compare()

lengths: min=81 med=138 max=250 n=5166
dropped 0 outlier cycles

=== Training LSTM model ===
Epoch 1/25


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3866 - loss: 1.2806 - val_accuracy: 0.7476 - val_loss: 0.6410
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7948 - loss: 0.5629 - val_accuracy: 0.8781 - val_loss: 0.3704
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9109 - loss: 0.2670 - val_accuracy: 0.8839 - val_loss: 0.3423
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9221 - loss: 0.2455 - val_accuracy: 0.9429 - val_loss: 0.1913
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9447 - loss: 0.1710 - val_accuracy: 0.9429 - val_loss: 0.1987
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9626 - loss: 0.1453 - val_accuracy: 0.9468 - val_loss: 0.1612
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9553 - loss: 0.1609 - val_accuracy: 0.9487 - val_loss: 0.1651
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9514 - loss: 0.1487 - val_accuracy: 0.9536 - val_

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.3275 - loss: 1.4153 - val_accuracy: 0.7021 - val_loss: 0.7924
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7265 - loss: 0.6438 - val_accuracy: 0.8694 - val_loss: 0.3528
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8965 - loss: 0.3050 - val_accuracy: 0.9043 - val_loss: 0.2731
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9391 - loss: 0.1885 - val_accuracy: 0.9323 - val_loss: 0.2170
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9429 - loss: 0.1777 - val_accuracy: 0.9304 - val_loss: 0.2007
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9540 - loss: 0.1448 - val_accuracy: 0.9342 - val_loss: 0.2083
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9666 - loss: 0.1083 - val_accuracy: 0.9371 - val_loss: 0.1914
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9600 - loss: 0.1249 - val_accuracy: 0.9439 - val_

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.6715 - loss: 2.3998 - val_accuracy: 0.9091 - val_loss: 0.3081
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8828 - loss: 0.3318 - val_accuracy: 0.9352 - val_loss: 0.2133
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9095 - loss: 0.2572 - val_accuracy: 0.9497 - val_loss: 0.2001
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9328 - loss: 0.2103 - val_accuracy: 0.9487 - val_loss: 0.1919
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9433 - loss: 0.1917 - val_accuracy: 0.9516 - val_loss: 0.1596
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9338 - loss: 0.2038 - val_accuracy: 0.9555 - val_loss: 0.1462
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9464 - loss: 0.1624 - val_accuracy: 0.9545 - val_loss: 0.1461
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9447 - loss: 0.1660 - val_accuracy: 0.9507 - val